## Backpropagation 
Manual implementation exercise (equivalent to loss.backward())

In [1]:
import torch
import torch.nn.functional as F
import matplotlib as plt
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
vocab_size = len(itos)

In [4]:
block_size = 3 # context length, how many chars we take to predict next

def build_dataset(words):
    X, Y = [], [] # input and labels
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix] # rolling window, crop and append new
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words) # mix up words randomly
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [5]:
# utility fcn for comparing manual vs torch gradient
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item() # manual vs torch grad exactly equal?
    app = torch.allclose(dt, t.grad) # manual vs torch grad approx equal?
    maxdiff = (dt - t.grad).abs().max().item() # largest difference 
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [6]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)

# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1

# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1

# BatchNorm parameters
bngain = torch.ones((1, n_hidden)) * 0.1 + 1.0
bnbias = torch.zeros((1, n_hidden)) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total

for p in parameters:
    p.requires_grad = True

4137


In [7]:
batch_size = 32
n = batch_size

# minibatch
ix = torch.randint(0, Xtr.shape[0], (n,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

In [8]:
emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors

# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation

# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias

# Non-linearity
h = torch.tanh(hpreact) # hidden layer

# Linear layer 2
logits = h @ W2 + b2 # output layer

# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3482, grad_fn=<NegBackward0>)

## Backprop 
Finding $\frac{d \text{loss}}{d \text{param}}=d\text{param}$. Size of gradient tensor should equal size of tensor itself (same .shape)

In [9]:

dlogprops = torch.zeros_like(logprobs) # array of zeros in logprops shape
dlogprops[range(n), Yb] = -1.0/n
cmp('logprobs', dlogprops, logprobs)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0


In [10]:
# dloss/dlogprops * dlogprops/dprops = -1/n * 1/props
dprobs = (1.0 / probs) * dlogprops
cmp('probs', dprobs, probs)

probs           | exact: True  | approximate: True  | maxdiff: 0.0


In [11]:
# to get probs, we first "extend" counts_sum_inv from 32x1 to 32x27 to apply elementwise to counts
# Derivative of element wise mul: couunts * dprobs
# Derivative of "extend": sum all the gradients involved in backprop
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0


In [12]:
dcounts = counts_sum_inv * dprobs # c_s_i broadcasts
# not yet corrects bc c_s_i depends on counts too
cmp('counts', dcounts, counts)

counts          | exact: False | approximate: False | maxdiff: 0.00623944029211998


In [13]:
dcounts_sum = (-counts_sum ** -2) * dcounts_sum_inv
cmp('counts_sum', dcounts_sum, counts_sum)

counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0


In [14]:
# counts: each row element of counts adds to give row of counts_sum
# 27 sums into 1
counts.shape, counts_sum.shape

(torch.Size([32, 27]), torch.Size([32, 1]))

In [15]:
dcounts += torch.ones_like(counts) * dcounts_sum # array of 1 in shape of counts (32x27)
# += bc dcounts was calculated before, so apply product rule
cmp('counts', dcounts, counts)

counts          | exact: True  | approximate: True  | maxdiff: 0.0


In [16]:
dnorm_logits = counts * dcounts
cmp('dnorm_logits', dnorm_logits, norm_logits)

dnorm_logits    | exact: True  | approximate: True  | maxdiff: 0.0


In [17]:
logits.shape, logit_maxes.shape # broadcasting in place

(torch.Size([32, 27]), torch.Size([32, 1]))

In [18]:
dlogits = dnorm_logits.clone()
dlogit_maxes = -dnorm_logits.sum(1, keepdim=True)
cmp('dlogit_maxes', dlogit_maxes, logit_maxes)

dlogit_maxes    | exact: True  | approximate: True  | maxdiff: 0.0


In [19]:
# similar to dlogprobs method above
# one_hot creates mask that is 1 at the max of each row
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
cmp('dlogits', dlogits, logits)

dlogits         | exact: True  | approximate: True  | maxdiff: 0.0


In [20]:
# Neural layer: matrix mul backprop: must make shapes align
dlogits.shape, h.shape, W2.shape, b2.shape

(torch.Size([32, 27]),
 torch.Size([32, 64]),
 torch.Size([64, 27]),
 torch.Size([27]))

In [21]:
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)

In [22]:
dhpreact = (1.0 - h ** 2) * dh

If dparam is broadcasted in process (has shape [1 x n] while its operation leads to [m x n]), we must sum the gradient.

A sum in forward pass always leads to broadcasting in backward pass. And vice versa

If forward pass is one value applied to many (broadcast), backward pass sums its gradients

If forward pass is many values to one (sum), backward pass broadcasts to shape of vector being summed up

In [23]:
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnraw = bngain * dhpreact
dbnbias = dhpreact.sum(0, keepdim=True)

In [ ]:
dbndiff = dbnraw * bnvar_inv # incorrect for now bc it is used elsewhere
dbnvar_inv = (dbnraw * bndiff).sum(0, keepdim=True)

In [39]:
dbnvar = (-0.5 * (bnvar + 1e-5) **(-1.5)) * dbnvar_inv

In [26]:
bnvar.shape, bndiff2.shape

(torch.Size([1, 64]), torch.Size([32, 64]))

In [31]:
dbndiff2 = (1.0/(n-1)) * torch.ones_like(bndiff2) * dbnvar

In [32]:
dbndiff += 2 * bndiff * dbndiff2

In [33]:
dhprebn = dbndiff.clone()
dbnmeani = (-dbndiff).sum(0, keepdim=True)
dhprebn += 1.0/n * (torch.ones_like(hprebn)) * dbnmeani

In [34]:
# hprebn = embcat @ W1 + b1
hprebn.shape, embcat.shape, W1.shape, b1.shape

(torch.Size([32, 64]),
 torch.Size([32, 30]),
 torch.Size([30, 64]),
 torch.Size([64]))

In [36]:
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)

In [37]:
# embcat = emb.view(emb.shape[0], -1)
demb = dembcat.view(emb.shape)

In [38]:
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k, j]
        dC[ix] += demb[k, j]